# 30 · Opto — uniform (expert) phase

## Setup

In [ ]:
%matplotlib inline
from shared_setup import *
from behav_utils.analysis.downsample import calculate_min_n, compute_ds_x
from behav_utils.analysis.stats_table import extract_stats


experiment, info = load_data()
print(f"Mode: {info['mode']}")

## Per Animal

In [ ]:
animal_id = 'SS16'
animal = experiment.get_animal(animal_id)
print(f"Animal: {animal.animal_id} ({animal.genotype.upper()})")

### Per Phase

#### Select Session & Trials

In [ ]:
distribution = 'Uniform'
session_type = 'masking'
phase_unfiltered = select_sessions(animal, distribution=distribution, session_type=session_type)
phase = filter_trials(phase_unfiltered, trial_type='non_opto')
phases_dist_sessionType = {tt: filter_trials(phase_unfiltered, trial_type=tt) for tt in ['non_opto', 'opto', 'post_opto']}

#### Compute Stats

##### Stats

In [ ]:
stat_names = ['accuracy', 'recency', 'win_stay', 'lose_shift', 'side_bias', 
			  'psychometric', 'psychometric_gof',
			  'hard_accuracy', 'easy_accuracy']

In [ ]:
stats_full_dist_sessionType = compare_phases(phases_dist_sessionType, stats=stat_names, downsample = False, 
				   n_permutations = 200, n_bootstrap = 200, 
				   reference='non_opto')

In [ ]:
stats_ds_dist_sessionType = compare_phases(phases_dist_sessionType, stats=stat_names, downsample = True,
						  n_permutations = 200, n_bootstrap = 200, reference = 'non_opto')

##### Update matrices

Full

In [ ]:
phase_opto_um_dist_sessionType = compute_um(phases_dist_sessionType['opto'])
phase_non_opto_um_dist_sessionType = compute_um(phases_dist_sessionType['non_opto'])
phase_post_opto_um_dist_sessionType = compute_um(phases_dist_sessionType['post_opto'])

Downsampled

In [ ]:
n = calculate_min_n([phases_dist_sessionType['opto'], phases_dist_sessionType['non_opto'], phases_dist_sessionType['post_opto']], unit = 'pairs')
phase_opto_ds_um_full_dist_sessionType = compute_ds_x(phases_dist_sessionType['opto'], 'um', n, n_repeats=100)   # 100 matched draws, aggregated
phase_non_opto_ds_um_full_dist_sessionType = compute_ds_x(phases_dist_sessionType['non_opto'], 'um', n, n_repeats=100)
phase_post_opto_ds_um_full_dist_sessionType = compute_ds_x(phases_dist_sessionType['post_opto'], 'um', n, n_repeats=100)
phase_opto_ds_um_dist_sessionType = phase_opto_ds_um_full_dist_sessionType['aggregated']
phase_non_opto_ds_um_dist_sessionType = phase_non_opto_ds_um_full_dist_sessionType['aggregated']
phase_post_opto_ds_um_dist_sessionType = phase_post_opto_ds_um_full_dist_sessionType['aggregated']

#### Plot Stats

##### Psychometrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
# Full
plot_psychometric(stats_full_dist_sessionType['phases']['non_opto']['psychometric'], title=f"Opto vs Nonopto trials ", ax = axes[0], color = PALETTE[0], 
				  label=f'Opto (n={stats_full_dist_sessionType["phases"]["non_opto"]["n_trials"]})')
plot_psychometric(stats_full_dist_sessionType['phases']['opto']['psychometric'], title=f"Opto vs Nonopto trials", ax = axes[0], color = PALETTE[1], 
				  label=f'Non-Opto (n={stats_full_dist_sessionType["phases"]["non_opto"]["n_trials"]})')
plot_psychometric(stats_full_dist_sessionType['phases']['post_opto']['psychometric'], title=f"Opto vs Nonopto trials", ax = axes[0], color = PALETTE[2], 
				  label=f'Post-Opto (n={stats_full_dist_sessionType["phases"]["post_opto"]["n_trials"]})')
axes[0].legend()

# Downsampled
plot_psychometric(stats_ds_dist_sessionType['phases']['opto']['psychometric'], title=f"Opto vs Nonopto trials (downsampled)", ax = axes[1], color = PALETTE[0], 
				  label=f'Opto (n={stats_ds_dist_sessionType["phases"]["opto"]["n_trials"]})')
plot_psychometric(stats_ds_dist_sessionType['phases']['non_opto']['psychometric'], title=f"Opto vs Nonopto trials (downsampled)", ax = axes[1], color = PALETTE[1], 
				  label=f'Non-Opto (n={stats_ds_dist_sessionType["phases"]["non_opto"]["n_trials"]})')
plot_psychometric(stats_ds_dist_sessionType['phases']['post_opto']['psychometric'], title=f"Opto vs Nonopto trials (downsampled)", ax = axes[1], color = PALETTE[2], 
				  label=f'Post-Opto (n={stats_ds_dist_sessionType["phases"]["post_opto"]["n_trials"]})')
axes[1].legend()

fig.suptitle(f"{animal.animal_id} - {animal.genotype.upper()} - {phases_dist_sessionType['non_opto'][0].session_type.capitalize()} Sessions - Psychometric Curves (n_sessions={stats_full_dist_sessionType['phases']['non_opto']['n_sessions']})", fontsize=14)
plt.tight_layout()

##### Update matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Full
axs = axes[0]
plot_um(phase_non_opto_um_dist_sessionType, title=f"Non-Opto trials (n_trials = {phase_non_opto_um_dist_sessionType['n_trials']})", ax = axs[0])
plot_um(phase_opto_um_dist_sessionType, title=f"Opto trials (n_trials = {phase_opto_um_dist_sessionType['n_trials']})", ax = axs[1])
plot_um(phase_post_opto_um_dist_sessionType, title=f"Post-Opto trials (n_trials = {phase_post_opto_um_dist_sessionType['n_trials']})", ax = axs[2])

# Downsampled
axs = axes[1]
plot_um(phase_non_opto_ds_um_dist_sessionType, title=f"Non-Opto trials - ds - (n_trials = {phase_non_opto_ds_um_dist_sessionType['n_trials']})", ax = axs[0])
plot_um(phase_opto_ds_um_dist_sessionType, title=f"Opto trials - ds - (n_trials = {phase_opto_ds_um_dist_sessionType['n_trials']})", ax = axs[1])
plot_um(phase_post_opto_ds_um_dist_sessionType, title=f"Post-Opto trials - ds - (n_trials = {phase_post_opto_ds_um_dist_sessionType['n_trials']})", ax = axs[2])


fig.suptitle(f"{animal.animal_id} - {animal.genotype.upper()} - {phases_dist_sessionType['opto'][0].session_type.capitalize()} - Update Matrix (n_sessions={len(phases_dist_sessionType['opto'])})", fontsize=14)
plt.tight_layout()

##### Stats

In [ ]:
from behav_utils.plotting.comparison import plot_stat_comparison

fig, axes = plot_stat_comparison(stats_full_dist_sessionType, ncols=4,
	suptitle=f"{animal.animal_id} — {animal.genotype.upper()} — {phases_dist_sessionType['opto'][0].session_type.capitalize()} — Summary Stats")


In [ ]:
fig, axes = plot_stat_comparison(stats_ds_dist_sessionType, ncols=4,
	suptitle=f"{animal.animal_id} — {animal.genotype.upper()} — Downsampled {phases_dist_sessionType['opto'][0].session_type.capitalize()} — Summary Stats")


### Phase Comparison

In [ ]:
from behav_utils.analysis.comparison import compute_interaction

distribution = 'Uniform'
session_type = 'masking'
phase_unfiltered = select_sessions(animal, distribution=distribution, session_type=session_type)
phases_dist_sessionType_1 = {tt: filter_trials(phase_unfiltered, trial_type=tt) for tt in ['non_opto', 'opto', 'post_opto']}

distribution = 'Uniform'
session_type = 'opto'
phase_unfiltered = select_sessions(animal, distribution=distribution, session_type=session_type)
phases_dist_sessionType_2 = {tt: filter_trials(phase_unfiltered, trial_type=tt) for tt in ['non_opto', 'opto', 'post_opto']}

distribution = 'Uniform'
session_type = 'alm_control_uni'
phase_unfiltered = select_sessions(animal, distribution=distribution, session_type=session_type)
phases_dist_sessionType_3 = {tt: filter_trials(phase_unfiltered, trial_type=tt) for tt in ['non_opto', 'opto', 'post_opto']}

res_mask = compare_phases(phases_dist_sessionType_1, stats=stat_names, reference='non_opto')
res_opto = compare_phases(phases_dist_sessionType_2, stats=stat_names, reference='non_opto')
res_alm_control = compare_phases(phases_dist_sessionType_3, stats=stat_names, reference='non_opto')

inter_opto = compute_interaction(res_mask, res_opto, contrast='opto_vs_non_opto', label_a = 'masking', label_b = 'opto')
inter_alm_control = compute_interaction(res_mask, res_alm_control, contrast='opto_vs_non_opto', label_a = 'masking', label_b = 'alm_control')

In [ ]:

from behav_utils.plotting.comparison import plot_interaction


for animal_id in ['SS14', 'SS15', 'SS16', 'SS17', 'SS18', 'SS21', 'SS22', 'SS23']:
	with PdfPages(OUT_DIR / f"{animal_id}_opto_alm_report.pdf") as pdf:
		print(f"Generating report for animal {animal_id}...")
		animal = experiment.get_animal(animal_id)
		distribution = 'Uniform'
		session_type = 'opto'
		phase_unfiltered = select_sessions(animal, distribution=distribution, session_type=session_type)
		phases_dist_sessionType_2 = {tt: filter_trials(phase_unfiltered, trial_type=tt) for tt in ['non_opto', 'opto', 'post_opto']}

		distribution = 'Uniform'
		session_type = 'alm_control_uni'
		phase_unfiltered = select_sessions(animal, distribution=distribution, session_type=session_type)
		phases_dist_sessionType_3 = {tt: filter_trials(phase_unfiltered, trial_type=tt) for tt in ['non_opto', 'opto', 'post_opto']}

		res_opto = compare_phases(phases_dist_sessionType_2, stats=stat_names, reference='non_opto')
		res_alm_control = compare_phases(phases_dist_sessionType_3, stats=stat_names, reference='non_opto')

		inter_alm_control = compute_interaction(res_opto, res_alm_control, contrast='opto_vs_non_opto', label_a = 'opto', label_b = 'alm_control')
		fig, axes = plot_interaction(inter_alm_control, ncols=4,
								suptitle=f"{animal.animal_id} — {animal.genotype.upper()} — opto vs alm_control sessions — Δ(opto − non_opto)")
		pdf.savefig(fig, dpi=300, bbox_inches='tight')
		plt.close(fig)

In [ ]:
from behav_utils.plotting.comparison import plot_interaction

fig, axes = plot_interaction(inter_opto, ncols=4,
	suptitle=f"{animal.animal_id} — {animal.genotype.upper()} — masking vs opto sessions — Δ(opto − non_opto)")

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
from behav_utils.plotting.comparison import plot_interaction, plot_stat_comparison
from behav_utils.analysis.comparison import compute_interaction

In [ ]:
OUT_DIR = FIG_DIR / 'animal_overview' / 'uniform'
OUT_DIR.mkdir(parents=True, exist_ok=True)

stat_names = ['accuracy', 'recency', 'win_stay', 'lose_shift', 'side_bias',
			  'psychometric', 'psychometric_gof',
			  'hard_accuracy', 'easy_accuracy']

animal_id_list = [
	'SS14', 'SS15', 'SS16', 'SS17', 'SS18', 
	# 'SS20', 
				  'SS21', 'SS22', 'SS23']
distribution_list = ['Uniform']
session_type_list = ['masking', 'opto', 'alm_control_uni']

N_PERM = 200
N_BOOT = 200

for animal_id in animal_id_list:
	animal = experiment.get_animal(animal_id)
	print(f"Animal: {animal.animal_id} ({animal.genotype.upper()})")

	with PdfPages(OUT_DIR / f"{animal.animal_id}_report.pdf") as pdf:
		for distribution in distribution_list:
			print(f"Distribution: {distribution}")

			# keep each session_type's result so the interaction section can
			# reuse it instead of recomputing compare_phases from scratch
			res_by_type = {}

			for session_type in session_type_list:
				print(f"Session Type: {session_type}")
				phase_unfiltered = select_sessions(animal, distribution=distribution,
												   session_type=session_type)
				phases = {tt: filter_trials(phase_unfiltered, trial_type=tt)
						  for tt in ['non_opto', 'opto', 'post_opto']}

				stats_full = compare_phases(phases, stats=stat_names, downsample=False,
											n_permutations=N_PERM, n_bootstrap=N_BOOT,
											reference='non_opto')
				stats_ds = compare_phases(phases, stats=stat_names, downsample=True,
										  n_permutations=N_PERM, n_bootstrap=N_BOOT,
										  reference='non_opto')
				res_by_type[session_type] = stats_full

				um_full = {tt: compute_um(phases[tt]) for tt in phases}
				n_pairs = calculate_min_n(list(phases.values()), unit='pairs')
				um_ds = {tt: compute_ds_x(phases[tt], 'um', n_pairs,
										  n_repeats=100)['aggregated'] for tt in phases}

				# ── psychometric curves ──────────────────────────────────
				# same order and colour mapping in both panels, and each label
				# takes its own phase's n
				fig, axes = plt.subplots(1, 2, figsize=(12, 6))
				for ax, res, tag in [(axes[0], stats_full, ''),
									 (axes[1], stats_ds, ' (downsampled)')]:
					for colour, tt, name in [(PALETTE[0], 'opto', 'Opto'),
											 (PALETTE[1], 'non_opto', 'Non-Opto'),
											 (PALETTE[2], 'post_opto', 'Post-Opto')]:
						plot_psychometric(
							res['phases'][tt]['psychometric'],
							title=f"Opto vs Nonopto trials{tag}", ax=ax, color=colour,
							label=f"{name} (n={res['phases'][tt]['n_trials']})")
					ax.legend()

				fig.suptitle(
					f"{animal.animal_id} - {animal.genotype.upper()} - "
					f"{session_type.capitalize()} Sessions - Psychometric Curves "
					f"(n_sessions={stats_full['phases']['non_opto']['n_sessions']})",
					fontsize=14)
				fig.tight_layout()
				pdf.savefig(fig, dpi=300, bbox_inches='tight')
				plt.close(fig)

				# ── update matrices ──────────────────────────────────────
				fig, axes = plt.subplots(2, 3, figsize=(18, 12))
				for row, (source, tag) in enumerate([(um_full, ''), (um_ds, ' - ds -')]):
					for col, tt in enumerate(['non_opto', 'opto', 'post_opto']):
						label = tt.replace('_', '-').title()
						plot_um(source[tt],
								title=f"{label} trials{tag} (n_trials = {source[tt]['n_trials']})",
								ax=axes[row][col])

				fig.suptitle(
					f"{animal.animal_id} - {animal.genotype.upper()} - "
					f"{session_type.capitalize()} - Update Matrix "
					f"(n_sessions={len(phases['opto'])})", fontsize=14)
				fig.tight_layout()
				pdf.savefig(fig, dpi=300, bbox_inches='tight')
				plt.close(fig)

				# ── summary stats ────────────────────────────────────────
				fig, axes = plot_stat_comparison(
					stats_full, ncols=4,
					suptitle=f"{animal.animal_id} — {animal.genotype.upper()} — "
							 f"{session_type.capitalize()} — Summary Stats")
				pdf.savefig(fig, dpi=300, bbox_inches='tight')
				plt.close(fig)

				fig, axes = plot_stat_comparison(
					stats_ds, ncols=4,
					suptitle=f"{animal.animal_id} — {animal.genotype.upper()} — "
							 f"Downsampled {session_type.capitalize()} — Summary Stats")
				pdf.savefig(fig, dpi=300, bbox_inches='tight')
				plt.close(fig)

			# ── interactions: reuse the results computed above ───────────
			print(f"Generating interaction plots for {animal.animal_id}")
			for other, label in [('opto', 'opto'), ('alm_control_uni', 'alm control')]:
				inter = compute_interaction(
					res_by_type['masking'], res_by_type[other],
					contrast='opto_vs_non_opto',
					label_a='masking', label_b=label.replace(' ', '_'))
				fig, axes = plot_interaction(
					inter, ncols=4,
					suptitle=f"{animal.animal_id} — {animal.genotype.upper()} — "
							 f"masking vs {label} sessions — Δ(opto − non_opto)")
				pdf.savefig(fig, dpi=300, bbox_inches='tight')
				plt.close(fig)

	print(f"Completed report for {animal.animal_id} ({animal.genotype.upper()})\n")


## Per Genotype